                               #######             
                              #                    
        ############ -#####+.##     ######         |	DataDrivenConstruction.io
        ##+########## .-##+ #-   .+########        |	
        ####     ####.     ..    #      ###        |	Dive into the future of flexibility in processing data 
        ####     .#+#.     #    .# .               |	without using BIM tools!
        ####     -#+#.    #.     .#.               |	
        ####      #+#.   ##-                       |	
        ####     ####        #.  #     -####       |	
        ############  .###+ #. ############        |
        #########.   +#### #+.   ########          |	If you have any questions, concerns or special requests,
                           #+                      |	please contact us at info@datadrivenconstruction.io	
                          #                        
                         #  



In [ ]:
#------------------------------------Input data-----------------------------------

import os
import sys
from pathlib import Path

# Folder of this repository (it holds DDC_WINDOWS_Converters and ddc-tools)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import convert

# Path address of the folder where Revit|IFC files are located
path = os.environ.get("DDC_BATCH", r'C:\Revit_Projects\Batch_Process_ChatGPT')
# Converter results and the flat tables (old layout, sheet Project_Data) go to <path>\converted
CONV = os.path.join(path, 'converted')

# Categories to analyse: Revit tables hold OST_ codes, IFC tables hold IFC classes
s_cat = ['OST_Walls', 'IfcWall', 'IfcWallStandardCase']

#-----------------------------------------------------------------------------------------

def convert_and_wait(file_path):
    """Runs the DDC converter for one file (RVTconverter.exe --properties --ifc, or the IFC to Excel converter)
    and builds <name>_rvt.xlsx / <name>_ifc.xlsx in CONV, the same as:  ddc convert <file> --xlsx <table>"""
    stem, ext = os.path.splitext(os.path.basename(file_path))
    kind = 'rvt' if ext.lower() in ('.rvt', '.rfa') else 'ifc'
    res = convert.convert(file_path, work=os.path.join(CONV, f'{stem}_{kind}'),
                          xlsx=os.path.join(CONV, f'{stem}_{kind}.xlsx'))
    print(f"Conversion done for project {stem}: {res['elements']} elements -> {res['xlsx']}")

# Ensure output directory exists
os.makedirs(CONV, exist_ok=True)
# Conversion process from RVT and IFC (a finished result folder is reused)
for file in os.listdir(path):
    if file.lower().endswith(('.ifc', '.rvt')):
        convert_and_wait(os.path.join(path, file))

#-------------------------------- Batch Processing ---------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt


def process_xlsx_files(folder_path, s_cat):
    for filename in os.listdir(folder_path):
        if filename.endswith((".xlsx")):
            file_path = os.path.join(folder_path, filename)

            # Load the XLSX file
            print(file_path)
            data = pd.read_excel(file_path, sheet_name="Project_Data")

# ==================================== < Chat_GPT > ==============================================


            # ================================
            # Code Block 1: Filtering data
            # ================================
            # Filtering data for the walls categories
            walls_data = data[data['Category'].isin(s_cat)]

            # Check if 'Type Name' and 'Width' columns are present
            if 'Type Name' in walls_data.columns and 'Width' in walls_data.columns:
                # Group by 'Type Name' and sum 'Width'
                grouped_walls_data = walls_data.groupby('Type Name')['Width'].sum()
            else:
                print("Either 'Type Name' or 'Width' column is missing in the data.")
                continue


            # ================================
            # Code Block 2: Creating a bar chart
            # ================================
            if not grouped_walls_data.empty:
            # Creating a horizontal bar chart
                plt.figure(figsize=(10, 8))
                grouped_walls_data.plot(kind='barh', color='teal')
                plt.title('Sum of Width by Type Name for ' + ', '.join(s_cat))
                plt.xlabel('Width')
                plt.ylabel('Type Name')
                plt.tight_layout()

                # Displaying the bar chart
                plt.savefig(os.path.join(folder_path, filename[:-5] + "_bar_chart.png"))
                plt.show()
                plt.close()


# ==================================== < / Chat_GPT > ==============================================


# Example call to the function
process_xlsx_files(CONV, s_cat)